# Baseline Evaluation - Production-Adapted News Summary

This notebook measures the baseline quality of one model with the current production-adapted prompt structure on `sunnysai12345/news-summary`.
Use the generated artifacts as the frozen baseline before creating a separate fine-tuning notebook.

Before running the evaluation cell, provide Kaggle credentials in one of these ways:
- set `KAGGLE_USERNAME` and `KAGGLE_KEY`, or
- place `kaggle.json` at `/content/drive/MyDrive/.kaggle/kaggle.json` after mounting Google Drive.


In [ ]:
import os
import subprocess
from pathlib import Path
import sys

IN_COLAB = False
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except Exception as exc:
    print(f'Running outside Colab: {exc}')

def find_repo_root(*starts: Path) -> Path | None:
    seen = set()
    for start in starts:
        if not start:
            continue
        for candidate in [start, *start.parents]:
            key = str(candidate.resolve()) if candidate.exists() else str(candidate)
            if key in seen:
                continue
            seen.add(key)
            if (candidate / 'reasoning_nlp').exists():
                return candidate
    return None

DEFAULT_DRIVE_ROOT = Path('/content/drive/MyDrive') if IN_COLAB else Path.cwd()
if IN_COLAB:
    REPO_DIR = Path('/content/video-summary')
    BRANCH_NAME = os.environ.get('VIDEO_SUMMARY_BRANCH', 'main').strip() or 'main'
    if not REPO_DIR.exists():
        subprocess.check_call([
            'git', 'clone', '--single-branch', '--branch', BRANCH_NAME,
            'https://github.com/TCTri205/video-summary.git', str(REPO_DIR)
        ])
    else:
        os.chdir(REPO_DIR)
        subprocess.check_call(['git', 'fetch', 'origin'])
        subprocess.check_call(['git', 'checkout', BRANCH_NAME])
        subprocess.check_call(['git', 'pull', 'origin', BRANCH_NAME])
    os.chdir(REPO_DIR)

SEARCH_STARTS = [
    Path.cwd(),
    Path(__file__).resolve().parent if '__file__' in globals() else None,
    Path('/content/video-summary') if IN_COLAB else None,
    Path('/content/drive/MyDrive/video-summary') if IN_COLAB else None,
]
REPO_ROOT = find_repo_root(*[path for path in SEARCH_STARTS if path is not None])
if REPO_ROOT is None:
    raise FileNotFoundError(
        'Cannot locate repo root containing reasoning_nlp. Clone or open the video-summary repo, '
        'or set the notebook working directory inside that repo before running imports.'
    )
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print('REPO_ROOT =', REPO_ROOT)
print('DEFAULT_DRIVE_ROOT =', DEFAULT_DRIVE_ROOT)


In [ ]:
import importlib.util
import subprocess
import sys

REQUIRED_PACKAGES = {
    'kaggle': 'kaggle',
    'pandas': 'pandas',
    'numpy': 'numpy',
    'matplotlib': 'matplotlib',
    'rouge_score': 'rouge-score',
    'transformers': 'transformers',
    'torch': 'torch',
    'accelerate': 'accelerate',
    'sentencepiece': 'sentencepiece',
    'bert_score': 'bert-score',
}
missing = [pip_name for module_name, pip_name in REQUIRED_PACKAGES.items() if importlib.util.find_spec(module_name) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
else:
    print('Python packages already satisfied')


In [ ]:
import os
import importlib
import inspect
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

import reasoning_nlp.eval.news_summary_baseline as news_summary_baseline_module
news_summary_baseline_module = importlib.reload(news_summary_baseline_module)
print('news_summary_baseline module =', inspect.getsourcefile(news_summary_baseline_module))

from reasoning_nlp.eval.news_summary_baseline import (
    NewsSummaryBaselineConfig,
    build_production_adapted_prompt_profile,
    run_baseline_evaluation,
    resolve_kaggle_cli_command,
)


def sanitize_model_label(value: str) -> str:
    cleaned = str(value).strip()
    for token in ('\\', '/', ':', ' '):
        cleaned = cleaned.replace(token, '_')
    return cleaned


def resolve_latest_finetuned_adapter(output_root: Path, runs_root: Path | None = None) -> tuple[Path, str]:
    return news_summary_baseline_module.resolve_latest_finetuned_adapter(
        finetune_output_root=output_root,
        finetune_runs_root=runs_root,
    )


def resolve_selected_model(
    *,
    use_finetuned_model: bool,
    base_model_name: str,
    finetune_output_root: Path,
    finetune_runs_root: Path,
    finetuned_model_path_override: str,
) -> tuple[str, str, str]:
    if not use_finetuned_model:
        return base_model_name, sanitize_model_label(base_model_name), 'base model'
    if finetuned_model_path_override.strip():
        adapter_dir = Path(finetuned_model_path_override.strip()).expanduser()
        adapter_config_path = adapter_dir / 'adapter_config.json'
        if not adapter_config_path.exists():
            raise FileNotFoundError(
                f'NEWS_SUMMARY_FINETUNED_MODEL_PATH points to {adapter_dir}, but adapter_config.json was not found.'
            )
        run_label = adapter_dir.parent.name if adapter_dir.name == 'adapter_model' else adapter_dir.name
        return str(adapter_dir.resolve()), sanitize_model_label(f'finetuned_{run_label}'), 'override path'
    adapter_dir, selection_note = resolve_latest_finetuned_adapter(finetune_output_root, finetune_runs_root)
    run_label = adapter_dir.parent.name if adapter_dir.name == 'adapter_model' else adapter_dir.name
    return str(adapter_dir), sanitize_model_label(f'finetuned_{run_label}'), selection_note


DATASET_SLUG = os.environ.get('KAGGLE_DATASET_SLUG', 'sunnysai12345/news-summary').strip()
CACHE_DIR = Path(os.environ.get('NEWS_SUMMARY_CACHE_DIR', str(DEFAULT_DRIVE_ROOT / 'video-summary-cache' / 'news-summary')))
RESULTS_DIR = Path(os.environ.get('NEWS_SUMMARY_RESULTS_DIR', str(DEFAULT_DRIVE_ROOT / 'video-summary-eval')))
KAGGLE_JSON_DRIVE_PATH = Path(os.environ.get('KAGGLE_JSON_DRIVE_PATH', str(DEFAULT_DRIVE_ROOT / '.kaggle' / 'kaggle.json')))

CSV_FILENAME = os.environ.get('NEWS_SUMMARY_CSV_FILENAME', 'news_summary.csv').strip()
ARTICLE_COLUMN = 'ctext'
SUMMARY_COLUMN = 'text'
AUX_HEADLINE_COLUMN = 'headlines'
SPLIT_COLUMN = os.environ.get('NEWS_SUMMARY_SPLIT_COLUMN', '').strip()
TARGET_SPLIT = os.environ.get('NEWS_SUMMARY_TARGET_SPLIT', 'test').strip()
USE_FIXED_SPLIT = os.environ.get('NEWS_SUMMARY_USE_FIXED_SPLIT', '1').strip().lower() not in {'0', 'false', 'no'}

# Chon model de danh gia:
# False = model chua fine-tune (mac dinh)
# True = model da fine-tune, tu dong lay adapter_model moi nhat trong NEWS_SUMMARY_FINETUNE_OUTPUT_ROOT
USE_FINETUNED_MODEL = False

BASE_MODEL_NAME = os.environ.get('VIDEO_SUMMARY_LOCAL_MODEL_VERSION', 'Qwen/Qwen2.5-3B-Instruct').strip()
FINETUNE_OUTPUT_ROOT = Path(
    os.environ.get('NEWS_SUMMARY_FINETUNE_OUTPUT_ROOT', str(DEFAULT_DRIVE_ROOT / 'video-summary-finetune'))
)
FINETUNE_CHECKPOINT_ROOT = Path(
    os.environ.get('NEWS_SUMMARY_FINETUNE_CHECKPOINT_ROOT', str(FINETUNE_OUTPUT_ROOT / 'runs'))
)
# Optional: dat duong dan adapter_model cu the neu muon khoa vao 1 run fine-tune.
# De chuoi rong '' de notebook tu tim run moi nhat.
FINETUNED_MODEL_PATH_OVERRIDE = os.environ.get('NEWS_SUMMARY_FINETUNED_MODEL_PATH', '').strip()
MODEL_NAME, SAFE_MODEL_NAME, MODEL_SELECTION_NOTE = resolve_selected_model(
    use_finetuned_model=USE_FINETUNED_MODEL,
    base_model_name=BASE_MODEL_NAME,
    finetune_output_root=FINETUNE_OUTPUT_ROOT,
    finetune_runs_root=FINETUNE_CHECKPOINT_ROOT,
    finetuned_model_path_override=FINETUNED_MODEL_PATH_OVERRIDE,
)

BACKEND = os.environ.get('VIDEO_SUMMARY_EVAL_BACKEND', 'local').strip().lower()
OPENAI_MODEL = os.environ.get('OPENAI_MODEL', '').strip()

EVAL_PROTOCOL_VERSION = os.environ.get('NEWS_SUMMARY_EVAL_PROTOCOL_VERSION', 'news-summary-baseline-v1').strip()
MAX_SAMPLES = int(os.environ.get('NEWS_SUMMARY_MAX_SAMPLES', '128'))
RANDOM_SEED = int(os.environ.get('NEWS_SUMMARY_RANDOM_SEED', '42'))
BATCH_SIZE = int(os.environ.get('NEWS_SUMMARY_BATCH_SIZE', '4'))
MAX_INPUT_CHARS = int(os.environ.get('NEWS_SUMMARY_MAX_INPUT_CHARS', '6000'))
MAX_INPUT_TOKENS = int(os.environ.get('NEWS_SUMMARY_MAX_INPUT_TOKENS', '3072'))
MAX_NEW_TOKENS = int(os.environ.get('NEWS_SUMMARY_MAX_NEW_TOKENS', '96'))
SPOTCHECK_SAMPLE_SIZE = int(os.environ.get('NEWS_SUMMARY_SPOTCHECK_SAMPLE_SIZE', '24'))
ENABLE_BERTSCORE = os.environ.get('NEWS_SUMMARY_ENABLE_BERTSCORE', '1').strip().lower() not in {'0', 'false', 'no'}
SAVE_PREDICTIONS_WITH_ARTICLE = os.environ.get('NEWS_SUMMARY_SAVE_PREDICTIONS_WITH_ARTICLE', '0').strip().lower() in {'1', 'true', 'yes'}
FORCE_REDOWNLOAD = os.environ.get('NEWS_SUMMARY_FORCE_REDOWNLOAD', '0').strip().lower() in {'1', 'true', 'yes'}

FROZEN_EVAL_IDS_PATH = Path(
    os.environ.get(
        'NEWS_SUMMARY_FROZEN_IDS_PATH',
        str(RESULTS_DIR / 'protocol' / f"{EVAL_PROTOCOL_VERSION}_{SAFE_MODEL_NAME}_frozen_eval_ids.csv"),
    )
)

config = NewsSummaryBaselineConfig(
    protocol_version=EVAL_PROTOCOL_VERSION,
    dataset_slug=DATASET_SLUG,
    cache_dir=CACHE_DIR,
    results_dir=RESULTS_DIR,
    kaggle_json_drive_path=KAGGLE_JSON_DRIVE_PATH,
    csv_filename=CSV_FILENAME,
    article_column=ARTICLE_COLUMN,
    summary_column=SUMMARY_COLUMN,
    aux_headline_column=AUX_HEADLINE_COLUMN,
    split_column=SPLIT_COLUMN,
    target_split=TARGET_SPLIT,
    use_fixed_split=USE_FIXED_SPLIT,
    frozen_eval_ids_path=FROZEN_EVAL_IDS_PATH,
    model_name=MODEL_NAME,
    backend=BACKEND,
    openai_model=OPENAI_MODEL,
    max_samples=MAX_SAMPLES,
    random_seed=RANDOM_SEED,
    batch_size=BATCH_SIZE,
    max_input_chars=MAX_INPUT_CHARS,
    max_input_tokens=MAX_INPUT_TOKENS,
    max_new_tokens=MAX_NEW_TOKENS,
    enable_bertscore=ENABLE_BERTSCORE,
    save_predictions_with_article=SAVE_PREDICTIONS_WITH_ARTICLE,
    spotcheck_sample_size=SPOTCHECK_SAMPLE_SIZE,
)
prompt_profile = build_production_adapted_prompt_profile()

print('USE_FINETUNED_MODEL =', USE_FINETUNED_MODEL)
print('MODEL_SELECTION_NOTE =', MODEL_SELECTION_NOTE)
print('MODEL_NAME =', MODEL_NAME)
print('FINETUNE_OUTPUT_ROOT =', FINETUNE_OUTPUT_ROOT)
print('FINETUNE_CHECKPOINT_ROOT =', FINETUNE_CHECKPOINT_ROOT)
print('BACKEND =', BACKEND)
print('DATASET_SLUG =', DATASET_SLUG)
print('KAGGLE_JSON_DRIVE_PATH =', KAGGLE_JSON_DRIVE_PATH)
print('FROZEN_EVAL_IDS_PATH =', FROZEN_EVAL_IDS_PATH)
display(Markdown(f'**Prompt profile:** `{prompt_profile.name}` | `{prompt_profile.prompt_version}`'))
display(Markdown(prompt_profile.adaptation_note))


In [ ]:
import json
import getpass
import shutil

def ensure_notebook_kaggle_credentials(kaggle_json_drive_path: Path) -> str:
    username = os.environ.get('KAGGLE_USERNAME', '').strip()
    key = os.environ.get('KAGGLE_KEY', '').strip()
    home_kaggle_dir = Path.home() / '.kaggle'
    home_kaggle_dir.mkdir(parents=True, exist_ok=True)
    home_kaggle_path = home_kaggle_dir / 'kaggle.json'

    print('Credential check:')
    print('  KAGGLE_JSON_DRIVE_PATH =', kaggle_json_drive_path)
    print('  kaggle.json exists in Drive =', kaggle_json_drive_path.exists())
    print('  env credentials set =', bool(username and key))

    if kaggle_json_drive_path.exists():
        home_kaggle_dir.mkdir(parents=True, exist_ok=True)
        shutil.copy2(kaggle_json_drive_path, home_kaggle_path)
        try:
            home_kaggle_path.chmod(0o600)
        except Exception:
            pass
        os.environ['KAGGLE_CONFIG_DIR'] = str(home_kaggle_dir)
        print('Using kaggle.json from Drive')
        return str(kaggle_json_drive_path)

    if username and key:
        kaggle_json_drive_path.parent.mkdir(parents=True, exist_ok=True)
        payload = {'username': username, 'key': key}
        kaggle_json_drive_path.write_text(json.dumps(payload), encoding='utf-8')
        try:
            kaggle_json_drive_path.chmod(0o600)
        except Exception:
            pass
        home_kaggle_path.write_text(json.dumps(payload), encoding='utf-8')
        try:
            home_kaggle_path.chmod(0o600)
        except Exception:
            pass
        os.environ['KAGGLE_CONFIG_DIR'] = str(home_kaggle_dir)
        print('Saved kaggle.json from environment variables')
        return str(kaggle_json_drive_path)

    if not IN_COLAB:
        raise FileNotFoundError(
            'Kaggle credential not found. Set KAGGLE_USERNAME/KAGGLE_KEY or place kaggle.json at '
            f'{kaggle_json_drive_path}.'
        )

    print('Kaggle credential not found. Enter credentials once to save them in Drive for future runs.')
    username = input('KAGGLE_USERNAME: ').strip()
    key = getpass.getpass('KAGGLE_KEY: ').strip()
    if not username or not key:
        raise FileNotFoundError(
            'Kaggle credential input was empty. Set KAGGLE_USERNAME/KAGGLE_KEY or place kaggle.json at '
            f'{kaggle_json_drive_path}.'
        )

    os.environ['KAGGLE_USERNAME'] = username
    os.environ['KAGGLE_KEY'] = key
    payload = {'username': username, 'key': key}
    kaggle_json_drive_path.parent.mkdir(parents=True, exist_ok=True)
    kaggle_json_drive_path.write_text(json.dumps(payload), encoding='utf-8')
    home_kaggle_path.write_text(json.dumps(payload), encoding='utf-8')
    for path in (kaggle_json_drive_path, home_kaggle_path):
        try:
            path.chmod(0o600)
        except Exception:
            pass
    os.environ['KAGGLE_CONFIG_DIR'] = str(home_kaggle_dir)
    print(f'Saved kaggle.json to {kaggle_json_drive_path}')
    return str(kaggle_json_drive_path)

credential_source = ensure_notebook_kaggle_credentials(KAGGLE_JSON_DRIVE_PATH)
kaggle_cli_command = resolve_kaggle_cli_command()
print('Kaggle credential ready from', credential_source)
print('Kaggle CLI command =', ' '.join(kaggle_cli_command))


In [ ]:
import reasoning_nlp.eval.news_summary_baseline as news_summary_baseline_module
importlib.reload(news_summary_baseline_module)
print('Running baseline eval from', inspect.getsourcefile(news_summary_baseline_module))

result = news_summary_baseline_module.run_baseline_evaluation(
    config=config,
    prompt_profile=prompt_profile,
    force_redownload=FORCE_REDOWNLOAD,
)
run_dir = result['run_dir']
metrics = result['metrics']
comparison_df = result['comparison_df']
per_example_df = result['per_example_df']
error_analysis_df = result['error_analysis_df']
spotcheck_df = result['spotcheck_df']

print('Artifacts saved to', run_dir)
display(pd.DataFrame([result['dataset_profile']]))
display(comparison_df)
display(error_analysis_df.head(10))
display(spotcheck_df.head(10))


In [ ]:
key_metrics = pd.DataFrame(
    [
        {'metric': 'rouge1', 'value': metrics.get('rouge1')},
        {'metric': 'rouge2', 'value': metrics.get('rouge2')},
        {'metric': 'rougeL', 'value': metrics.get('rougeL')},
        {'metric': 'bertscore_f1', 'value': metrics.get('bertscore_f1')},
        {'metric': 'parse_success_rate', 'value': metrics.get('parse_success_rate')},
        {'metric': 'hallucination_proxy_rate', 'value': metrics.get('hallucination_proxy_rate')},
        {'metric': 'instruction_leakage_rate', 'value': metrics.get('instruction_leakage_rate')},
    ]
)
display(key_metrics)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].bar(key_metrics['metric'], key_metrics['value'].fillna(0.0))
axes[0].set_title('Key automatic metrics')
axes[0].tick_params(axis='x', rotation=45)

per_example_ok = per_example_df.loc[per_example_df['status'].eq('ok')].copy()
if not per_example_ok.empty:
    axes[1].scatter(per_example_ok['latency_ms'], per_example_ok['rougeL'], alpha=0.6)
    axes[1].set_title('Latency vs ROUGE-L')
    axes[1].set_xlabel('latency_ms')
    axes[1].set_ylabel('rougeL')
else:
    axes[1].set_title('No scored examples')
plt.tight_layout()
plt.show()


In [ ]:
report_path = run_dir / 'baseline_report.md'
display(Markdown(report_path.read_text(encoding='utf-8')))


In [ ]:
HUMAN_EVAL_PATH = Path(os.environ.get('NEWS_SUMMARY_HUMAN_EVAL_PATH', '')).expanduser() if os.environ.get('NEWS_SUMMARY_HUMAN_EVAL_PATH', '').strip() else None
if HUMAN_EVAL_PATH and HUMAN_EVAL_PATH.exists():
    human_df = pd.read_csv(HUMAN_EVAL_PATH)
    summary = {
        'faithfulness_mean': float(pd.to_numeric(human_df['faithfulness_score'], errors='coerce').mean()),
        'coverage_mean': float(pd.to_numeric(human_df['coverage_score'], errors='coerce').mean()),
        'fluency_mean': float(pd.to_numeric(human_df['fluency_score'], errors='coerce').mean()),
        'hallucination_rate': float(human_df['hallucination_flag'].astype(str).str.strip().str.lower().isin({'1', 'true', 'yes', 'y'}).mean()),
        'usable_for_training_target_rate': float(human_df['usable_for_training_target'].astype(str).str.strip().str.lower().isin({'1', 'true', 'yes', 'y'}).mean()),
    }
    summary_df = pd.DataFrame([summary])
    summary_df.to_csv(run_dir / 'human_eval_summary.csv', index=False)
    display(summary_df)
else:
    print('Optional: set NEWS_SUMMARY_HUMAN_EVAL_PATH to a filled spotcheck CSV to aggregate human evaluation.')


In [ ]:
import json
import reasoning_nlp.eval.news_summary_finetune as news_summary_finetune_module

news_summary_finetune_module = importlib.reload(news_summary_finetune_module)
from reasoning_nlp.eval.news_summary_finetune import build_before_after_comparison


def load_json_file(path: Path) -> dict:
    return json.loads(path.read_text(encoding='utf-8'))


def resolve_latest_post_train_metrics(post_eval_dir: Path) -> Path:
    metric_candidates = [path for path in post_eval_dir.rglob('metrics.json') if path.is_file()]
    if not metric_candidates:
        raise FileNotFoundError(
            f'No post-train metrics.json found under {post_eval_dir}. Run the post-train eval cell in '
            'notebooks/model_finetune_news_summary_colab.ipynb first.'
        )
    return max(metric_candidates, key=lambda path: (path.stat().st_mtime, str(path)))


def resolve_latest_finetune_run(runs_root: Path) -> tuple[Path, Path, Path]:
    if not runs_root.exists():
        raise FileNotFoundError(
            f'Fine-tune runs root not found: {runs_root}. Run notebooks/model_finetune_news_summary_colab.ipynb first '
            'or set NEWS_SUMMARY_FINETUNE_CHECKPOINT_ROOT.'
        )
    candidates = []
    for run_dir in runs_root.iterdir():
        if not run_dir.is_dir():
            continue
        training_manifest_path = run_dir / 'training_manifest.json'
        post_eval_dir = run_dir / 'eval_after_train'
        if not training_manifest_path.exists() or not post_eval_dir.exists():
            continue
        try:
            metrics_path = resolve_latest_post_train_metrics(post_eval_dir)
        except FileNotFoundError:
            continue
        candidates.append((training_manifest_path.stat().st_mtime, run_dir, training_manifest_path, metrics_path))
    if not candidates:
        raise FileNotFoundError(
            f'No fine-tune run with training_manifest.json and post-train metrics found under {runs_root}.'
        )
    _, run_dir, training_manifest_path, metrics_path = max(candidates, key=lambda item: (item[0], item[1].name))
    return run_dir.resolve(), training_manifest_path.resolve(), metrics_path.resolve()


def resolve_latest_baseline_manifest(results_root: Path) -> Path:
    if not results_root.exists():
        raise FileNotFoundError(
            f'Baseline results root not found: {results_root}. Run this notebook once with USE_FINETUNED_MODEL = False first.'
        )
    candidates = [path for path in results_root.rglob('baseline_manifest.json') if path.is_file()]
    if not candidates:
        raise FileNotFoundError(
            f'No baseline_manifest.json found under {results_root}. Run this notebook once with USE_FINETUNED_MODEL = False first.'
        )
    return max(candidates, key=lambda path: (path.stat().st_mtime, str(path))).resolve()


COMPARE_BASELINE_MANIFEST_PATH = Path(os.environ.get('NEWS_SUMMARY_COMPARE_BASELINE_MANIFEST_PATH', '')).expanduser() if os.environ.get('NEWS_SUMMARY_COMPARE_BASELINE_MANIFEST_PATH', '').strip() else None
COMPARE_TRAINING_MANIFEST_PATH = Path(os.environ.get('NEWS_SUMMARY_COMPARE_TRAINING_MANIFEST_PATH', '')).expanduser() if os.environ.get('NEWS_SUMMARY_COMPARE_TRAINING_MANIFEST_PATH', '').strip() else None
COMPARE_FINETUNE_RUNS_ROOT = Path(
    os.environ.get('NEWS_SUMMARY_FINETUNE_CHECKPOINT_ROOT', str(FINETUNE_OUTPUT_ROOT / 'runs'))
)

if COMPARE_TRAINING_MANIFEST_PATH is not None:
    if not COMPARE_TRAINING_MANIFEST_PATH.exists():
        raise FileNotFoundError(
            f'NEWS_SUMMARY_COMPARE_TRAINING_MANIFEST_PATH does not exist: {COMPARE_TRAINING_MANIFEST_PATH}'
        )
    finetune_run_dir = COMPARE_TRAINING_MANIFEST_PATH.parent.resolve()
    training_manifest_path = COMPARE_TRAINING_MANIFEST_PATH.resolve()
    training_manifest = load_json_file(training_manifest_path)
    post_eval_dir = Path(str(training_manifest.get('paths', {}).get('post_eval_dir', finetune_run_dir / 'eval_after_train')))
    finetuned_metrics_path = resolve_latest_post_train_metrics(post_eval_dir)
else:
    finetune_run_dir, training_manifest_path, finetuned_metrics_path = resolve_latest_finetune_run(COMPARE_FINETUNE_RUNS_ROOT)
    training_manifest = load_json_file(training_manifest_path)

baseline_manifest_path = COMPARE_BASELINE_MANIFEST_PATH
if baseline_manifest_path is None and 'result' in globals() and isinstance(result, dict) and not USE_FINETUNED_MODEL:
    candidate = Path(result['run_dir']) / 'baseline_manifest.json'
    if candidate.exists():
        baseline_manifest_path = candidate.resolve()
if baseline_manifest_path is None:
    baseline_manifest_raw = str(training_manifest.get('baseline_manifest_path', '')).strip()
    if baseline_manifest_raw:
        candidate = Path(baseline_manifest_raw).expanduser()
        if candidate.exists():
            baseline_manifest_path = candidate.resolve()
if baseline_manifest_path is None:
    baseline_manifest_path = resolve_latest_baseline_manifest(RESULTS_DIR)

baseline_manifest = load_json_file(baseline_manifest_path)
baseline_metrics = baseline_manifest.get('metrics', {})
finetuned_metrics = load_json_file(finetuned_metrics_path)
comparison_df = build_before_after_comparison(
    baseline_metrics=baseline_metrics,
    finetuned_metrics=finetuned_metrics,
)

comparison_output_path = finetune_run_dir / 'before_after_comparison_from_eval_notebook.csv'
comparison_df.to_csv(comparison_output_path, index=False)

artifact_df = pd.DataFrame(
    [
        {
            'baseline_manifest_path': str(baseline_manifest_path),
            'training_manifest_path': str(training_manifest_path),
            'finetuned_metrics_path': str(finetuned_metrics_path),
            'comparison_output_path': str(comparison_output_path),
        }
    ]
)
display(artifact_df)

focus_metrics = [
    'rouge1',
    'rouge2',
    'rougeL',
    'bertscore_f1',
    'parse_success_rate',
    'hallucination_proxy_rate',
    'instruction_leakage_rate',
    'avg_latency_ms',
]
focus_df = comparison_df.loc[comparison_df['metric'].isin(focus_metrics)].copy()
focus_df['metric'] = pd.Categorical(focus_df['metric'], categories=focus_metrics, ordered=True)
focus_df = focus_df.sort_values('metric').reset_index(drop=True)
display(focus_df)

if not focus_df.empty:
    x_positions = list(range(len(focus_df)))
    width = 0.35
    fig, axes = plt.subplots(1, 2, figsize=(16, 4))
    axes[0].bar(
        [x - width / 2 for x in x_positions],
        pd.to_numeric(focus_df['baseline'], errors='coerce').fillna(0.0),
        width=width,
        label='baseline',
    )
    axes[0].bar(
        [x + width / 2 for x in x_positions],
        pd.to_numeric(focus_df['finetuned'], errors='coerce').fillna(0.0),
        width=width,
        label='finetuned',
    )
    axes[0].set_title('Baseline vs finetuned')
    axes[0].set_xticks(x_positions)
    axes[0].set_xticklabels([str(value) for value in focus_df['metric']], rotation=45, ha='right')
    axes[0].legend()

    delta_values = pd.to_numeric(focus_df['delta'], errors='coerce').fillna(0.0)
    delta_colors = ['#2e7d32' if value >= 0 else '#c62828' for value in delta_values]
    axes[1].bar([str(value) for value in focus_df['metric']], delta_values, color=delta_colors)
    axes[1].axhline(0.0, color='black', linewidth=1)
    axes[1].set_title('Finetuned - baseline delta')
    axes[1].tick_params(axis='x', rotation=45)
    plt.tight_layout()
    plt.show()
else:
    print('No overlapping focus metrics available for comparison.')
